In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.model_selection import train_test_split

# Load dataset
file_path = r"/content/drive/My Drive/bug/categorized_bug_reports_dynamic_5000.csv"
df = pd.read_csv(file_path)

Load Dataset & Generate Embeddings


In [ ]:
import numpy as np
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
# Convert bug descriptions into embeddings
descriptions = df["Short Description"].astype(str).tolist()
embeddings = sbert_model.encode(descriptions, convert_to_numpy=True)

# Convert category names to numerical labels
df["Label"] = df["Category"].astype("category").cat.codes
labels = df["Label"].values

# Save embeddings and labels for future use
np.save("bug_embeddings.npy", embeddings)
np.save("bug_labels.npy", labels)
print("✔️ Embeddings generated and saved!")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✔️ Embeddings generated and saved!


In [ ]:
ls

bug_embeddings.npy  bug_labels.npy  drive/  sample_data/


In [ ]:
bug = np.load("bug_embeddings.npy")

In [ ]:
labels = np.load("bug_labels.npy")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
dataset = BugDataset(bug, labels)

In [ ]:
for anchor, positive, negative in dataloader:
    print("Anchor shape:", anchor.shape)   # Expected: (32, 768)
    print("Positive shape:", positive.shape) # Expected: (32, 768)
    print("Negative shape:", negative.shape) # Expected: (32, 768)
    break

Anchor shape: torch.Size([32, 384])
Positive shape: torch.Size([32, 384])
Negative shape: torch.Size([32, 384])


In [ ]:
num_samples = 1000
embedding_dim = 384

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ==============================
# 🔹 Siamese Network
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super(SiameseNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, embedding_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.normalize(x, p=2, dim=1)  # L2 normalization
        return x  # Embedding output

# ==============================
# 🔹 Contrastive Loss Function
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = torch.norm(anchor - positive, p=2, dim=1) ** 2
        neg_dist = torch.norm(anchor - negative, p=2, dim=1) ** 2
        loss = torch.mean(F.relu(pos_dist - neg_dist + self.margin))  # Hinge loss
        return loss

# ==============================
# 🔹 Custom Dataset for Training
# ==============================
class BugDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.scaler = StandardScaler()
        self.embeddings = self.scaler.fit_transform(embeddings)  # Normalize features
        self.labels = labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, index):
        anchor = self.embeddings[index]
        anchor_label = self.labels[index]

        positive_indices = np.where(self.labels == anchor_label)[0]
        positive = self.embeddings[np.random.choice(positive_indices)]

        negative_indices = np.where(self.labels != anchor_label)[0]
        negative = self.embeddings[np.random.choice(negative_indices)]

        return (
            torch.tensor(anchor, dtype=torch.float32),
            torch.tensor(positive, dtype=torch.float32),
            torch.tensor(negative, dtype=torch.float32)
        )

# ==============================
# 🔹 Load Dataset
# ==============================


dataset = BugDataset(bug, labels)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# ==============================
# 🔹 Train Siamese Network
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiameseNetwork(input_dim=384, embedding_dim=128).to(device)
criterion = ContrastiveLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

num_epochs = 10  # Train for a fixed number of epochs

for epoch in range(num_epochs):
    total_loss = 0.0
    for anchor, positive, negative in dataloader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        optimizer.zero_grad()
        anchor_embed = model(anchor)
        positive_embed = model(positive)
        negative_embed = model(negative)

        loss = criterion(anchor_embed, positive_embed, negative_embed)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("✔️ Siamese Network training complete!")


Epoch [1/10], Loss: 0.3691
Epoch [2/10], Loss: 0.2329
Epoch [3/10], Loss: 0.1896
Epoch [4/10], Loss: 0.1622
Epoch [5/10], Loss: 0.1464
Epoch [6/10], Loss: 0.1314
Epoch [7/10], Loss: 0.1199
Epoch [8/10], Loss: 0.1104
Epoch [9/10], Loss: 0.1000
Epoch [10/10], Loss: 0.0913
✔️ Siamese Network training complete!


In [ ]:
import numpy as np

# Save to Drive
np.save('/content/drive/My Drive/bug/embeddings.npy', embeddings)

print("✔️ NumPy array saved!")


✔️ NumPy array saved!


In [ ]:
save_path = "/content/drive/MyDrive/bug/siamese_model.pth"  # Adjust path as needed
torch.save(model.state_dict(), save_path)
print(f"✔ Model saved at {save_path}")


✔ Model saved at /content/drive/MyDrive/bug/siamese_model.pth
